# Laboratorio — Métricas y Validación de Modelos
## Unidad 2 · Lab 1 · Duración: 1 hora · Modalidad: individual o parejas

**versión: 2025-1  |  modificado: 2026-04-06**

---

<div style="background-color:#F8F9FA; border:2px solid #AEB6BF;
     padding:12px 18px; border-radius:8px; margin:10px 0;">
<strong>🎓 Modo de uso:</strong> Este notebook es compartido por dos cursos.<br><br>
<span style="color:#2E86C1; font-weight:bold;">🔵 Pregrado</span>
— Trabaja el contenido general y los bloques azules. Los bloques amarillos
son opcionales y te darán contexto adicional.<br><br>
<span style="color:#B7950B; font-weight:bold;">🟡 Doctorado</span>
— Trabaja el contenido general y <em>ambos</em> bloques. Los bloques azules
te ofrecen la intuición; los amarillos, la formalización.
</div>

---

### 📋 Instrucciones por audiencia

| | Pregrado | Doctorado |
|--|----------|-----------|
| **Obligatorio** | Partes 1, 2, 3 + preguntas azules | Todo lo anterior + TODOs [PhD] + preguntas amarillas |
| **Opcional** | Bonus azul | Bonus amarillo |
| **Entrega** | .ipynb ejecutado | .ipynb ejecutado |

---

### 🗺️ Estructura del laboratorio

| Parte | Tema | Tiempo | Audiencia |
|-------|------|--------|-----------|
| 1 | Ejercicio sin computador: matriz de confusión a mano | 10 min | Todos |
| 2 | Métricas de clasificación y ROC | 20 min | Todos |
| 3 | Validación cruzada con pipeline | 15 min | Todos |
| 4 | Análisis e interpretación | 10 min | Todos |
| Bonus | Curvas de aprendizaje y nested CV | Opcional | — |


In [2]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 100

# dataset: wine (sklearn built-in)  |  generado sintéticamente: no
# Usaremos Wine como clasificación multiclase (3 clases)
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='variedad')

# Split estratificado (ANTES de cualquier transformación)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Función auxiliar: visualizar matriz de confusión
def plot_cm(cm, labels, title='Matriz de Confusión', ax=None):
    """Visualiza una matriz de confusión normalizada por fila."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, ax=ax, cbar=False)
    ax.set_xlabel('Predicción', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')

print('✅ Setup completo')
print(f'   numpy {np.__version__} · pandas {pd.__version__} · scikit-learn {sklearn_version}')
print(f'   Wine dataset — train: {X_train.shape}, test: {X_test.shape}')
print(f'   Clases: {wine.target_names}')

✅ Setup completo
   numpy 2.4.4 · pandas 3.0.2 · scikit-learn 1.8.0
   Wine dataset — train: (133, 13), test: (45, 13)
   Clases: ['class_0' 'class_1' 'class_2']


---
## PARTE 1 — Ejercicio sin computador (10 min) ✏️

**Instrucción:** Responde en tu cuaderno ANTES de ejecutar la celda de verificación.

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado</span><br><br>

Un detector de fraude en transacciones bancarias tiene estos resultados sobre 10 casos de test:

| Caso | Real (y) | Predicho (ŷ) |
|------|----------|-------------|
| 1 | fraude | no-fraude |
| 2 | no-fraude | no-fraude |
| 3 | fraude | fraude |
| 4 | no-fraude | fraude |
| 5 | fraude | fraude |
| 6 | no-fraude | no-fraude |
| 7 | fraude | fraude |
| 8 | no-fraude | no-fraude |
| 9 | no-fraude | fraude |
| 10 | fraude | no-fraude |

**a)** Construye la matriz de confusión (fraude = positivo).  
**b)** Calcula Accuracy, Precision, Recall y F1 **a mano**.  
**c)** En el contexto de detección de fraude, ¿qué tipo de error es más grave: FP o FN? Justifica.

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres ver la versión formal? Continúa en el bloque 🟡 Doctorado.</span>
</div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado</span><br><br>

Usa los mismos datos del bloque azul y además:

**d)** Expresa el costo total de los errores como:  
$$\text{Costo} = C_{FN} \cdot FN + C_{FP} \cdot FP$$
Si $C_{FN} = 1000$ (fraude no detectado) y $C_{FP} = 50$ (falsa alarma), ¿cuál es el costo total?

**e)** Deriva analíticamente el umbral óptimo $\theta^*$ que minimiza el costo esperado, dado que el modelo entrega $\hat{P}(y=1|x)$. Pista: iguala las derivadas del costo respecto a $\theta$.

**f)** ¿Cuál es la relación entre $\theta^*$ y el ratio de costos $C_{FP}/C_{FN}$?

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de esta derivación está en el bloque 🔵 Pregrado.</span>
</div>

In [ ]:
# ── Verificación Parte 1 — Ejecutar DESPUÉS de responder en el cuaderno ──
y_real_p1 = np.array([1, 0, 1, 0, 1, 0, 1, 0, 0, 1])  # 1=fraude, 0=no-fraude
y_pred_p1 = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 0])

cm_p1 = confusion_matrix(y_real_p1, y_pred_p1)
print('=== Verificación Parte 1 ===')
print(f'Matriz de Confusión:\n{cm_p1}')
print(f'Accuracy : {accuracy_score(y_real_p1, y_pred_p1):.3f}')
print(f'Precision: {precision_score(y_real_p1, y_pred_p1):.3f}')
print(f'Recall   : {recall_score(y_real_p1, y_pred_p1):.3f}')
print(f'F1       : {f1_score(y_real_p1, y_pred_p1):.3f}')

# [PhD] Costo total
C_FN = 1000
C_FP = 50
tn, fp, fn, tp = cm_p1.ravel()
costo = C_FN * fn + C_FP * fp
print(f'\n[PhD] Costo total: {C_FN}*{fn} + {C_FP}*{fp} = {costo}')

=== Verificación Parte 1 ===
Matriz de Confusión:
[[3 3]
 [2 2]]
Accuracy : 0.500
Precision: 0.400
Recall   : 0.500
F1       : 0.444

[PhD] Costo total: 1000*2 + 50*3 = 2150


---
## PARTE 2 — Métricas de clasificación y ROC (20 min)

In [ ]:
# ━━━ PARTE 2: IMPLEMENTACIÓN ━━━

# TODO 1: Crea un Pipeline con StandardScaler + LogisticRegression (max_iter=2000)
#         y entrenarlo con X_train, y_train
# Pista: Pipeline([('scaler', ...), ('clf', ...)])

pipeline = None  # Reemplaza con tu pipeline
# Tu código aquí:


In [ ]:
# TODO 2: Genera predicciones y_pred con X_test
# Pista: usa pipeline.predict()

y_pred = None  # Reemplaza
# Tu código aquí:


In [ ]:
# TODO 3: Calcula e imprime el classification_report completo
#         Pista: classification_report(y_test, y_pred, target_names=wine.target_names)

# Tu código aquí:


In [ ]:
# TODO 4: Visualiza la matriz de confusión usando la función plot_cm del setup
#         y el pipeline ya entrenado.

# Tu código aquí:


In [ ]:
# TODO 5 [PhD]: Calcula el AUC-OVR (One-vs-Rest) para las 3 clases de Wine
#               Pista: usa pipeline.predict_proba() y roc_auc_score con multi_class='ovr'

# Tu código aquí:


In [ ]:
# ─── TESTS DE SANIDAD — PARTE 2 — NO MODIFICAR ───
print('=== Tests de Sanidad — Parte 2 ===')
try:
    assert pipeline is not None, 'pipeline no está definido'
    print('✅ Pipeline definido')
except AssertionError as e:
    print(f'❌ {e} — Completa el TODO 1 primero')

try:
    assert y_pred is not None, 'y_pred no está definido'
    assert len(y_pred) == len(y_test), 'y_pred debe tener el mismo tamaño que y_test'
    print(f'✅ y_pred tiene el tamaño correcto ({len(y_pred)} muestras)')
except AssertionError as e:
    print(f'❌ {e}')

try:
    assert set(y_pred) <= {0, 1, 2}, 'y_pred contiene clases fuera del rango esperado'
    acc = accuracy_score(y_test, y_pred)
    assert 0.0 <= acc <= 1.0, 'Accuracy fuera de rango [0,1]'
    print(f'✅ Accuracy en rango válido: {acc:.3f}')
except AssertionError as e:
    print(f'❌ {e}')

try:
    from sklearn.pipeline import Pipeline as SKPipeline
    assert isinstance(pipeline, SKPipeline), 'El pipeline debe ser un objeto sklearn.pipeline.Pipeline'
    print('✅ Pipeline es instancia de sklearn.pipeline.Pipeline')
except AssertionError as e:
    print(f'❌ {e}')

try:
    scaler_in_pipe = pipeline.named_steps.get('scaler', None)
    assert scaler_in_pipe is not None, 'El pipeline debe incluir un paso llamado scaler'
    # Verificar que el scaler NO fue fiteado en test (data leakage check)
    assert hasattr(scaler_in_pipe, 'mean_'), 'El scaler debe estar fiteado'
    print('✅ Scaler en pipeline (no data leakage)')
except AssertionError as e:
    print(f'❌ {e}')

---
## PARTE 3 — Validación Cruzada con Pipeline (15 min)

In [ ]:
# TODO 6: Evalúa el pipeline con Stratified 5-Fold CV usando cross_val_score
#         Métrica: 'f1_weighted' (apropiada para multiclase)
#         Usa TODOS los datos (X, y), no solo el split de train.
# Pista: cross_val_score(pipeline, X, y, cv=StratifiedKFold(...), scoring='f1_weighted')

cv_scores = None  # Reemplaza con el array de scores por fold
# Tu código aquí:


In [ ]:
# TODO 7: Imprime el F1 promedio ± desviación estándar y visualiza
#         la distribución de scores por fold con un barplot o boxplot.

# Tu código aquí:


In [ ]:
# TODO 8: Compara el pipeline de Regresión Logística con un DecisionTreeClassifier
#         (max_depth=5) usando la misma Stratified 5-Fold CV.
#         Imprime ambos resultados y comenta cuál es mejor.

# Tu código aquí:


In [ ]:
# TODO 9 [PhD]: Implementa Nested CV para la Regresión Logística
#               Loop externo: StratifiedKFold(5)
#               Loop interno: GridSearchCV sobre C = logspace(-3, 3, 7)
#               Compara el F1 con y sin nested CV.
# Pista: cross_val_score(GridSearchCV(pipeline, param_grid, cv=inner_cv), X, y, cv=outer_cv)

# Tu código aquí:


In [ ]:
# ─── TESTS DE SANIDAD — PARTE 3 — NO MODIFICAR ───
print('=== Tests de Sanidad — Parte 3 ===')
try:
    assert cv_scores is not None, 'cv_scores no está definido'
    assert len(cv_scores) == 5, f'Debe haber 5 scores (uno por fold), hay {len(cv_scores)}'
    print(f'✅ cv_scores tiene 5 folds: {np.round(cv_scores, 3)}')
except AssertionError as e:
    print(f'❌ {e}')

try:
    assert all(0.0 <= s <= 1.0 for s in cv_scores), 'Algún score está fuera de [0,1]'
    print(f'✅ Todos los scores en rango válido [0, 1]')
except (AssertionError, TypeError) as e:
    print(f'❌ {e}')

try:
    mean_score = np.mean(cv_scores)
    assert mean_score > 0.5, f'F1 promedio ({mean_score:.3f}) demasiado bajo — revisa tu implementación'
    print(f'✅ F1 promedio razonable: {mean_score:.3f}')
except (AssertionError, TypeError) as e:
    print(f'❌ {e}')

---
## PARTE 4 — Análisis e Interpretación (10 min)

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado</span><br><br>

Responde las siguientes preguntas en celdas Markdown a continuación:

1. ¿Por qué es importante usar un Pipeline (con el scaler dentro) al hacer validación cruzada, en lugar de normalizar los datos antes de la CV?

2. Si la desviación estándar de tus scores CV es muy alta (ej. ±0.15), ¿qué podría explicarlo? ¿Qué harías para reducirla?

3. Imagina que debes presentar el rendimiento de tu clasificador de vinos a alguien no técnico (un sommelier). ¿Cómo explicarías el F1-weighted de forma comprensible?

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres profundizar? Continúa en el bloque 🟡 Doctorado.</span>
</div>

<!-- RESPUESTA MODELO: 
P1: Si normalizas antes de CV, el scaler "ve" los datos de test (media y std calculadas con todo), introduciendo data leakage optimista. / Dentro del Pipeline, el fit del scaler ocurre solo en el fold de train. / Esto puede inflar métricas en datasets pequeños o con alta correlación.
P2: Dataset pequeño (poca muestra por fold) / Alta variabilidad intrínseca de los datos / Considerar más folds (k=10) o repetir CV con distintos seeds.
P3: "De cada 100 vinos, nuestro sistema identifica correctamente la variedad de unos X, balanceando los errores de confusión entre las tres variedades." / Mostrar la matriz de confusión como tabla de errores concretos.
-->

**Respuesta 1:**

*(escribe aquí)*

**Respuesta 2:**

*(escribe aquí)*

**Respuesta 3:**

*(escribe aquí)*

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado</span><br><br>

4. Supón que los folds de CV no son independientes (datos con correlación temporal o espacial). ¿Qué sesgo introduce esto en la estimación del error? ¿Qué estrategia de CV usarías?

5. En el experimento de Nested CV, ¿observaste diferencia entre el F1 con y sin nested CV? ¿A qué atribuyes esa diferencia y en qué dirección va el sesgo?

6. Diseña un experimento para cuantificar el sesgo de selección de modelos usando el dataset Wine: varía el número de hiperparámetros en el grid y mide cómo cambia la brecha entre CV simple y nested CV.

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de esta derivación está en el bloque 🔵 Pregrado.</span>
</div>

<!-- RESPUESTA MODELO:
P4: Correlación temporal → los folds comparten información, estimación demasiado optimista. / Usar TimeSeriesSplit o GroupKFold. / La CV estándar asume i.i.d., violación → intervalos de confianza incorrectos.
P5: Nested CV generalmente da F1 ligeramente menor / El sesgo es optimista en CV simple porque el mismo dato se usa para seleccionar y evaluar. / A mayor número de hiperparámetros, mayor sesgo de selección.
P6: Variar grid size de 1 a 50 hiperparámetros, graficar F1_simple - F1_nested vs grid_size. / Hipótesis: brecha crece con tamaño del grid. / Repetir con múltiples seeds para estimar varianza.
-->

**Respuesta 4 [PhD]:**

*(escribe aquí)*

**Respuesta 5 [PhD]:**

*(escribe aquí)*

**Respuesta 6 [PhD]:**

*(escribe aquí)*

---
## BONUS

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Bonus Pregrado</span><br><br>

Grafica la **curva de aprendizaje** del pipeline de Regresión Logística sobre el dataset Wine.
- Usa `learning_curve` de sklearn con Stratified 5-Fold.
- Interpreta: ¿el modelo tiene alto bias, alta varianza, o está bien calibrado? ¿Más datos ayudarían?

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres ir más lejos? Continúa en el bloque 🟡 Doctorado.</span>
</div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Bonus Doctorado</span><br><br>

Implementa un **test de hipótesis para comparar dos clasificadores** (LR vs DT) sobre Wine:
- Usa el test de McNemar sobre las predicciones de ambos modelos en el conjunto de test.
- Interpreta el p-valor: ¿son significativamente diferentes?
- Referencia: Dietterich (1998). *Approximate statistical tests for comparing supervised classification learning algorithms*. Neural Computation.

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de esta derivación está en el bloque 🔵 Pregrado.</span>
</div>

In [ ]:
# Espacio de trabajo para el Bonus


---
## ✅ Checklist de entrega

### Pregrado
- [ ] Parte 1: respondí en el cuaderno antes de ejecutar
- [ ] TODO 1–4 completados y celdas ejecutadas
- [ ] TODO 6–8 completados con visualización
- [ ] Tests de sanidad pasan (✅ PASS)
- [ ] Parte 4: preguntas 1–3 respondidas

### Doctorado (adicional)
- [ ] TODO 5: AUC-OVR calculado
- [ ] TODO 9: Nested CV implementado y comparado
- [ ] Parte 4: preguntas 4–6 respondidas
- [ ] Bonus completado

### Ambas audiencias
- [ ] El notebook ejecuta de arriba a abajo sin errores
- [ ] Entrego el archivo con mi nombre: `ML_U2_Lab01_Apellido_Nombre.ipynb`